# What Makes a Society Healthy?

## A Machine Learning Analysis of Life Expectancy Across Countries

## 1. Objective and Research Question

### Research Question

Which economic, healthcare, lifestyle, environmental and social factors best explain differences in life expectancy across countries?

### Objective

The objective of this project is to analyze the determinants of life expectancy across countries using international datasets and machine learning methods.

The project compares different groups of explanatory variables, including economic development, healthcare spending, lifestyle risk factors, environmental exposure and social indicators.

The goal is not only to build a predictive model, but also to understand which macro-categories of variables are most informative in explaining cross-country differences in life expectancy.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

## 2. Data Sources and Variable Framework

The analysis uses country-level international data from public sources.

The target variable is:

- **Life expectancy at birth**

The explanatory variables are organized into five macro-categories:

1. **Economic Factors**
   - GDP per capita

2. **Healthcare Factors**
   - Health expenditure as a percentage of GDP
   - Health expenditure per capita

3. **Lifestyle Factors**
   - Obesity prevalence
   - Smoking prevalence

4. **Environmental Factors**
   - PM2.5 air pollution

5. **Social Factors**
   - Education

Most variables refer to 2022. However, PM2.5 air pollution data were not available for 2022 in the selected World Bank indicator, so 2020 PM2.5 exposure was used as the most recent available environmental proxy with broad country coverage.

In [2]:
base_dataset = pd.read_csv("../data/life_expectancy_dataset_v2.csv")
model_audit = pd.read_csv("../data/model_audit.csv")
rf_performance_audit = pd.read_csv("../data/random_forest_performance_audit.csv")
feature_importance_audit = pd.read_csv("../data/feature_importance_audit.csv")
final_model_dataset = pd.read_csv("../data/final_absolute_healthcare_environment_model_dataset.csv")

print("Base dataset shape:", base_dataset.shape)
print("Final model dataset shape:", final_model_dataset.shape)

base_dataset.head()

Base dataset shape: (212, 6)
Final model dataset shape: (155, 9)


,country_code,country,year,life_expectancy,gdp_per_capita,health_expenditure
0,EAR,Early-demographic dividend,2022,71.430455,4043.827763,4.780528
1,EAS,East Asia & Pacific,2022,76.676717,13139.210981,6.996939
2,FCS,Fragile and conflict affected situations,2022,62.702272,1850.917431,4.400807
3,HPC,Heavily indebted poor countries (HIPC),2022,63.840466,1121.483873,4.664174
4,LTE,Late-demographic dividend,2022,76.954367,12286.084681,6.165431


## 3. Data Cleaning and Preparation

The original project involved collecting, merging and cleaning multiple country-level indicators.

The main cleaning and preparation steps were:

- selecting observations for the target year where available;
- converting downloaded indicator values to numeric format;
- merging variables by country code;
- removing observations with missing values depending on each model specification;
- checking for implausible life expectancy values;
- excluding anomalous values below 40 years of life expectancy;
- testing education as a social factor, while noting its limited country coverage;
- testing air pollution as an environmental factor, using 2020 PM2.5 exposure because 2022 data were unavailable;
- comparing two healthcare spending measures: health expenditure as a percentage of GDP and health expenditure per capita.

The final modeling dataset contains the variables used in the best-performing model.

In [3]:
# Missing values in the base dataset

base_missing = base_dataset.isna().sum().to_frame("missing_values")
base_missing["missing_percentage"] = (
    base_missing["missing_values"] / len(base_dataset) * 100
).round(2)

base_missing

,missing_values,missing_percentage
country_code,0,0.00
country,0,0.00
year,0,0.00
life_expectancy,0,0.00
gdp_per_capita,0,0.00
health_expenditure,21,9.91


In [4]:
# Distribution of life expectancy in the base dataset

base_dataset["life_expectancy"].describe()

count    212.000000
mean      73.236561
std        7.055553
min       53.931000
25%       67.638250
50%       74.238000
75%       78.284250
max       85.746000
Name: life_expectancy, dtype: float64

In [5]:
# Check implausible life expectancy values

base_dataset[
    base_dataset["life_expectancy"] < 40
][
    ["country_code", "country", "year", "life_expectancy"]
]

,country_code,country,year,life_expectancy


The data inspection identified implausible life expectancy values below 40 years.

These observations were excluded from the modeling datasets because they are likely to reflect data quality issues or exceptional anomalies rather than stable cross-country health patterns.

This cleaning choice helps avoid distorted model results caused by extreme anomalous values.

In [6]:
# Final model dataset used by the best-performing model

print("Final model dataset observations:", len(final_model_dataset))

final_model_dataset[
    [
        "country_code",
        "country",
        "life_expectancy",
        "gdp_per_capita",
        "health_expenditure_per_capita",
        "obesity",
        "smoking",
        "air_pollution"
    ]
].head()

Final model dataset observations: 155


,country_code,country,life_expectancy,gdp_per_capita,health_expenditure_per_capita,obesity,smoking,air_pollution
0,AFG,Afghanistan,65.617,357.261153,80.651604,19.222592,22.7,46.087094
1,ALB,Albania,78.769,7756.961887,506.869202,23.360185,21.9,15.707004
2,DZA,Algeria,76.129,4960.303343,180.334549,23.814993,21.2,25.552656
3,AND,Andorra,84.016,42414.047986,3190.113281,18.099177,36.3,9.080281
4,ARG,Argentina,75.806,13962.189409,1415.795776,35.355195,23.8,14.908174


In [7]:
# Missing values in the final model dataset

final_model_dataset[
    [
        "life_expectancy",
        "gdp_per_capita",
        "health_expenditure_per_capita",
        "obesity",
        "smoking",
        "air_pollution"
    ]
].isna().sum()

life_expectancy                  0
gdp_per_capita                   0
health_expenditure_per_capita    0
obesity                          0
smoking                          0
air_pollution                    0
dtype: int64

In [8]:
aggregate_keywords = [
    "World",
    "income",
    "IDA",
    "IBRD",
    "OECD",
    "Euro area",
    "Arab",
    "East Asia",
    "Europe",
    "Central Asia",
    "Latin America",
    "Caribbean",
    "Middle East",
    "North Africa",
    "South Asia",
    "Sub-Saharan",
    "Fragile",
    "demographic dividend",
    "Heavily indebted",
    "Small states",
    "Pacific island",
    "Least developed",
    "European Union",
    "Africa Eastern",
    "Africa Western"
]

base_aggregate_check = base_dataset[
    base_dataset["country"].str.contains(
        "|".join(aggregate_keywords),
        case=False,
        na=False
    )
]

final_aggregate_check = final_model_dataset[
    final_model_dataset["country"].str.contains(
        "|".join(aggregate_keywords),
        case=False,
        na=False
    )
]

print("Potential aggregate observations in base dataset:", len(base_aggregate_check))
print("Potential aggregate observations in final model dataset:", len(final_aggregate_check))

final_aggregate_check[["country_code", "country"]].head(30)

Potential aggregate observations in base dataset: 10
Potential aggregate observations in final model dataset: 0


,country_code,country


# Full Project Audit Before Final Notebook Reconstruction

This section checks data quality, dataset consistency, model outputs and final files before rebuilding the final notebook.

In [9]:
import pandas as pd
import numpy as np
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Load core files
base_dataset = pd.read_csv("../data/life_expectancy_dataset_v2.csv")
model_audit = pd.read_csv("../data/model_audit.csv")
model_comparison = pd.read_csv("../data/model_comparison.csv")
rf_performance_audit = pd.read_csv("../data/random_forest_performance_audit.csv")
feature_importance_audit = pd.read_csv("../data/feature_importance_audit.csv")
final_model_dataset = pd.read_csv("../data/final_absolute_healthcare_environment_model_dataset.csv")
final_m9_importance = pd.read_csv("../data/final_absolute_healthcare_environment_feature_importance.csv")

print("Files loaded successfully.")
print()
print("Base dataset shape:", base_dataset.shape)
print("Final model dataset shape:", final_model_dataset.shape)
print("Model audit shape:", model_audit.shape)
print("RF performance audit shape:", rf_performance_audit.shape)
print("Feature importance audit shape:", feature_importance_audit.shape)
print("Final M9 feature importance shape:", final_m9_importance.shape)

Files loaded successfully.

Base dataset shape: (212, 6)
Final model dataset shape: (155, 9)
Model audit shape: (9, 9)
RF performance audit shape: (7, 6)
Feature importance audit shape: (26, 4)
Final M9 feature importance shape: (5, 2)


In [10]:
print("Base dataset columns:")
print(base_dataset.columns.tolist())

print("\nFinal model dataset columns:")
print(final_model_dataset.columns.tolist())

print("\nModel audit columns:")
print(model_audit.columns.tolist())

print("\nRF performance audit columns:")
print(rf_performance_audit.columns.tolist())

print("\nFeature importance audit columns:")
print(feature_importance_audit.columns.tolist())

Base dataset columns:
['country_code', 'country', 'year', 'life_expectancy', 'gdp_per_capita', 'health_expenditure']

Final model dataset columns:
['country_code', 'country', 'life_expectancy', 'gdp_per_capita', 'health_expenditure', 'obesity', 'smoking', 'air_pollution', 'health_expenditure_per_capita']

Model audit columns:
['Model_ID', 'Model_name', 'Algorithm', 'Macro_categories', 'Variables_used', 'Observations', 'R_squared', 'MAE_years', 'Notes']

RF performance audit columns:
['Model_ID', 'Model_name', 'Observations', 'R_squared', 'MAE_years', 'Features']

Feature importance audit columns:
['Model_ID', 'Model_name', 'Feature', 'Importance']


In [11]:
print("Missing values - base dataset")
display(base_dataset.isna().sum().to_frame("missing_values"))

print("\nMissing values - final model dataset")
display(final_model_dataset.isna().sum().to_frame("missing_values"))

print("\nMissing values - model audit")
display(model_audit.isna().sum().to_frame("missing_values"))

print("\nMissing values - feature importance audit")
display(feature_importance_audit.isna().sum().to_frame("missing_values"))

Missing values - base dataset


,missing_values
country_code,0
country,0
year,0
life_expectancy,0
gdp_per_capita,0
health_expenditure,21



Missing values - final model dataset


,missing_values
country_code,0
country,0
life_expectancy,0
gdp_per_capita,0
health_expenditure,0
obesity,0
smoking,0
air_pollution,0
health_expenditure_per_capita,0



Missing values - model audit


,missing_values
Model_ID,0
Model_name,0
Algorithm,0
Macro_categories,0
Variables_used,0
Observations,0
R_squared,0
MAE_years,0
Notes,0



Missing values - feature importance audit


,missing_values
Model_ID,0
Model_name,0
Feature,0
Importance,0


In [12]:
aggregate_keywords = [
    "World",
    "income",
    "IDA",
    "IBRD",
    "OECD",
    "Euro area",
    "Arab",
    "East Asia",
    "Europe",
    "Central Asia",
    "Latin America",
    "Caribbean",
    "Middle East",
    "North Africa",
    "South Asia",
    "Sub-Saharan",
    "Fragile",
    "demographic dividend",
    "Heavily indebted",
    "Small states",
    "Pacific island",
    "Least developed",
    "European Union",
    "Africa Eastern",
    "Africa Western"
]

pattern = "|".join(aggregate_keywords)

base_aggregate_check = base_dataset[
    base_dataset["country"].str.contains(pattern, case=False, na=False)
]

final_aggregate_check = final_model_dataset[
    final_model_dataset["country"].str.contains(pattern, case=False, na=False)
]

print("Potential aggregate observations in base dataset:", len(base_aggregate_check))
display(base_aggregate_check[["country_code", "country"]].head(50))

print("\nPotential aggregate observations in final model dataset:", len(final_aggregate_check))
display(final_aggregate_check[["country_code", "country"]].head(50))

Potential aggregate observations in base dataset: 10


,country_code,country
0,EAR,Early-demographic dividend
1,EAS,East Asia & Pacific
2,FCS,Fragile and conflict affected situations
3,HPC,Heavily indebted poor countries (HIPC)
4,LTE,Late-demographic dividend
5,LCN,Latin America & Caribbean
6,LDC,Least developed countries: UN classification
8,PST,Post-demographic dividend
9,PRE,Pre-demographic dividend
10,SAS,South Asia



Potential aggregate observations in final model dataset: 0


,country_code,country


In [13]:
print("Life expectancy summary - base dataset")
display(base_dataset["life_expectancy"].describe())

print("\nLife expectancy summary - final model dataset")
display(final_model_dataset["life_expectancy"].describe())

print("\nBase dataset observations with life expectancy < 40")
display(
    base_dataset[
        base_dataset["life_expectancy"] < 40
    ][["country_code", "country", "year", "life_expectancy"]]
)

print("\nFinal model dataset observations with life expectancy < 40")
display(
    final_model_dataset[
        final_model_dataset["life_expectancy"] < 40
    ][["country_code", "country", "life_expectancy"]]
)

Life expectancy summary - base dataset


count    212.000000
mean      73.236561
std        7.055553
min       53.931000
25%       67.638250
50%       74.238000
75%       78.284250
max       85.746000
Name: life_expectancy, dtype: float64


Life expectancy summary - final model dataset


count    155.000000
mean      72.971255
std        7.032692
min       54.079000
25%       67.425500
50%       74.125000
75%       78.102500
max       84.016000
Name: life_expectancy, dtype: float64


Base dataset observations with life expectancy < 40


,country_code,country,year,life_expectancy



Final model dataset observations with life expectancy < 40


,country_code,country,life_expectancy


In [14]:
m9_expected_columns = [
    "country_code",
    "country",
    "life_expectancy",
    "gdp_per_capita",
    "health_expenditure_per_capita",
    "obesity",
    "smoking",
    "air_pollution"
]

print("Expected M9 columns present?")
for col in m9_expected_columns:
    print(col, "->", col in final_model_dataset.columns)

display(
    final_model_dataset[m9_expected_columns].describe().round(3)
)

Expected M9 columns present?
country_code -> True
country -> True
life_expectancy -> True
gdp_per_capita -> True
health_expenditure_per_capita -> True
obesity -> True
smoking -> True
air_pollution -> True


,life_expectancy,gdp_per_capita,health_expenditure_per_capita,obesity,smoking,air_pollution
count,155.000,155.000,155.000,155.000,155.000,155.000
mean,72.971,17368.051,1374.840,22.532,19.977,23.483
std,7.033,23826.468,2043.510,12.761,10.060,15.657
min,54.079,302.993,15.858,2.016,3.300,4.895
25%,67.426,2368.593,102.196,12.497,11.250,12.299
50%,74.125,6680.445,469.349,21.784,19.500,19.494
75%,78.102,21102.159,1642.432,29.637,27.450,28.613
max,84.016,123719.659,10930.140,71.650,48.300,85.122


In [15]:
print("Model audit:")
display(model_audit)

print("\nModel IDs:")
print(model_audit["Model_ID"].tolist())

print("\nDuplicate model IDs:")
display(model_audit[model_audit["Model_ID"].duplicated(keep=False)])

print("\nBest model by R²:")
display(model_audit.sort_values("R_squared", ascending=False).head(3))

print("\nBest model by MAE:")
display(model_audit.sort_values("MAE_years", ascending=True).head(3))

Model audit:


,Model_ID,Model_name,Algorithm,Macro_categories,Variables_used,Observations,R_squared,MAE_years,Notes
0,M1,Baseline Linear Regression,Linear Regression,"Economic, Relative Healthcare","gdp_per_capita, health_expenditure (% of GDP)",191,0.447,4.59,Baseline model using economic development and relative healthcare spending.
1,M2,Log-GDP Linear Regression,Linear Regression,"Economic, Relative Healthcare","log_gdp_per_capita, health_expenditure (% of GDP)",191,0.789,2.80,Tests diminishing returns of income by applying a logarithmic transformation to GDP per capita.
2,M3,Baseline Random Forest,Random Forest,"Economic, Relative Healthcare","gdp_per_capita, health_expenditure",191,0.757,2.75,Non-linear baseline model using the same variables as M1.
3,M4,Social Extension Random Forest,Random Forest,"Economic, Relative Healthcare, Social","gdp_per_capita, health_expenditure, education",107,0.630,2.58,Adds education as a social factor; sample size decreases substantially due to missing data.
4,M5,Obesity Extension Random Forest,Random Forest,"Economic, Relative Healthcare, Lifestyle","gdp_per_capita, health_expenditure, obesity",177,0.732,2.86,Adds obesity as the first lifestyle variable.
5,M6,Lifestyle Extension Random Forest,Random Forest,"Economic, Relative Healthcare, Lifestyle","gdp_per_capita, health_expenditure, obesity, smoking",155,0.802,2.77,Adds obesity and smoking; smoking improves predictive performance.
6,M7,Absolute Healthcare Spending Random Forest,Random Forest,"Economic, Absolute Healthcare, Lifestyle","gdp_per_capita, health_expenditure_per_capita, obesity, smoking",155,0.787,2.79,Replaces health expenditure as % of GDP with health expenditure per capita.
7,M8,Lifestyle & Environment Random Forest,Random Forest,"Economic, Relative Healthcare, Lifestyle, Environment","gdp_per_capita, health_expenditure, obesity, smoking, air_pollution",155,0.812,2.70,Adds PM2.5 air pollution using 2020 data as the most recent available environmental proxy.
8,M9,Absolute Healthcare & Environment Random Forest,Random Forest,"Economic, Absolute Healthcare, Lifestyle, Environment","gdp_per_capita, health_expenditure_per_capita, obesity, smoking, air_pollution",155,0.822,2.59,"Final candidate model combining economic development, absolute healthcare spending, lifestyle factors and air pollution."



Model IDs:
['M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9']

Duplicate model IDs:


,Model_ID,Model_name,Algorithm,Macro_categories,Variables_used,Observations,R_squared,MAE_years,Notes



Best model by R²:


,Model_ID,Model_name,Algorithm,Macro_categories,Variables_used,Observations,R_squared,MAE_years,Notes
8,M9,Absolute Healthcare & Environment Random Forest,Random Forest,"Economic, Absolute Healthcare, Lifestyle, Environment","gdp_per_capita, health_expenditure_per_capita, obesity, smoking, air_pollution",155,0.822,2.59,"Final candidate model combining economic development, absolute healthcare spending, lifestyle factors and air pollution."
7,M8,Lifestyle & Environment Random Forest,Random Forest,"Economic, Relative Healthcare, Lifestyle, Environment","gdp_per_capita, health_expenditure, obesity, smoking, air_pollution",155,0.812,2.70,Adds PM2.5 air pollution using 2020 data as the most recent available environmental proxy.
5,M6,Lifestyle Extension Random Forest,Random Forest,"Economic, Relative Healthcare, Lifestyle","gdp_per_capita, health_expenditure, obesity, smoking",155,0.802,2.77,Adds obesity and smoking; smoking improves predictive performance.



Best model by MAE:


,Model_ID,Model_name,Algorithm,Macro_categories,Variables_used,Observations,R_squared,MAE_years,Notes
3,M4,Social Extension Random Forest,Random Forest,"Economic, Relative Healthcare, Social","gdp_per_capita, health_expenditure, education",107,0.630,2.58,Adds education as a social factor; sample size decreases substantially due to missing data.
8,M9,Absolute Healthcare & Environment Random Forest,Random Forest,"Economic, Absolute Healthcare, Lifestyle, Environment","gdp_per_capita, health_expenditure_per_capita, obesity, smoking, air_pollution",155,0.822,2.59,"Final candidate model combining economic development, absolute healthcare spending, lifestyle factors and air pollution."
7,M8,Lifestyle & Environment Random Forest,Random Forest,"Economic, Relative Healthcare, Lifestyle, Environment","gdp_per_capita, health_expenditure, obesity, smoking, air_pollution",155,0.812,2.70,Adds PM2.5 air pollution using 2020 data as the most recent available environmental proxy.


In [16]:
print("model_audit equals model_comparison?")
print(model_audit.equals(model_comparison))

if not model_audit.equals(model_comparison):
    print("\nDifferences may be due to column order or formatting. Showing both heads:")
    display(model_audit.head())
    display(model_comparison.head())

model_audit equals model_comparison?
True


In [17]:
print("Random Forest performance audit:")
display(rf_performance_audit)

print("\nRandom Forest models sorted by R²:")
display(rf_performance_audit.sort_values("R_squared", ascending=False))

print("\nRandom Forest models sorted by MAE:")
display(rf_performance_audit.sort_values("MAE_years", ascending=True))

Random Forest performance audit:


,Model_ID,Model_name,Observations,R_squared,MAE_years,Features
0,M3,Baseline Random Forest,191,0.757,2.75,"gdp_per_capita, health_expenditure"
1,M4,Social Extension Random Forest,107,0.630,2.58,"gdp_per_capita, health_expenditure, education"
2,M5,Obesity Extension Random Forest,177,0.732,2.86,"gdp_per_capita, health_expenditure, obesity"
3,M6,Lifestyle Extension Random Forest,155,0.802,2.77,"gdp_per_capita, health_expenditure, obesity, smoking"
4,M7,Absolute Healthcare Spending Random Forest,155,0.787,2.79,"gdp_per_capita, health_expenditure_per_capita, obesity, smoking"
5,M8,Lifestyle & Environment Random Forest,155,0.812,2.70,"gdp_per_capita, health_expenditure, obesity, smoking, air_pollution"
6,M9,Absolute Healthcare & Environment Random Forest,155,0.822,2.59,"gdp_per_capita, health_expenditure_per_capita, obesity, smoking, air_pollution"



Random Forest models sorted by R²:


,Model_ID,Model_name,Observations,R_squared,MAE_years,Features
6,M9,Absolute Healthcare & Environment Random Forest,155,0.822,2.59,"gdp_per_capita, health_expenditure_per_capita, obesity, smoking, air_pollution"
5,M8,Lifestyle & Environment Random Forest,155,0.812,2.70,"gdp_per_capita, health_expenditure, obesity, smoking, air_pollution"
3,M6,Lifestyle Extension Random Forest,155,0.802,2.77,"gdp_per_capita, health_expenditure, obesity, smoking"
4,M7,Absolute Healthcare Spending Random Forest,155,0.787,2.79,"gdp_per_capita, health_expenditure_per_capita, obesity, smoking"
0,M3,Baseline Random Forest,191,0.757,2.75,"gdp_per_capita, health_expenditure"
2,M5,Obesity Extension Random Forest,177,0.732,2.86,"gdp_per_capita, health_expenditure, obesity"
1,M4,Social Extension Random Forest,107,0.630,2.58,"gdp_per_capita, health_expenditure, education"



Random Forest models sorted by MAE:


,Model_ID,Model_name,Observations,R_squared,MAE_years,Features
1,M4,Social Extension Random Forest,107,0.630,2.58,"gdp_per_capita, health_expenditure, education"
6,M9,Absolute Healthcare & Environment Random Forest,155,0.822,2.59,"gdp_per_capita, health_expenditure_per_capita, obesity, smoking, air_pollution"
5,M8,Lifestyle & Environment Random Forest,155,0.812,2.70,"gdp_per_capita, health_expenditure, obesity, smoking, air_pollution"
0,M3,Baseline Random Forest,191,0.757,2.75,"gdp_per_capita, health_expenditure"
3,M6,Lifestyle Extension Random Forest,155,0.802,2.77,"gdp_per_capita, health_expenditure, obesity, smoking"
4,M7,Absolute Healthcare Spending Random Forest,155,0.787,2.79,"gdp_per_capita, health_expenditure_per_capita, obesity, smoking"
2,M5,Obesity Extension Random Forest,177,0.732,2.86,"gdp_per_capita, health_expenditure, obesity"


In [18]:
print("Feature importance audit:")
display(
    feature_importance_audit.sort_values(
        by=["Model_ID", "Importance"],
        ascending=[True, False]
    )
)

print("\nFeature importance sums by model:")
importance_sums = feature_importance_audit.groupby("Model_ID")["Importance"].sum()
display(importance_sums)

print("\nCheck if importance sums are approximately 1:")
display(np.isclose(importance_sums, 1.0))

Feature importance audit:


,Model_ID,Model_name,Feature,Importance
0,M3,Baseline Random Forest,gdp_per_capita,0.832213
1,M3,Baseline Random Forest,health_expenditure,0.167787
2,M4,Social Extension Random Forest,gdp_per_capita,0.661767
3,M4,Social Extension Random Forest,education,0.294693
4,M4,Social Extension Random Forest,health_expenditure,0.043540
5,M5,Obesity Extension Random Forest,gdp_per_capita,0.807329
6,M5,Obesity Extension Random Forest,health_expenditure,0.096621
7,M5,Obesity Extension Random Forest,obesity,0.096050
8,M6,Lifestyle Extension Random Forest,gdp_per_capita,0.784144
9,M6,Lifestyle Extension Random Forest,smoking,0.093903



Feature importance sums by model:


Model_ID
M3    1.0
M4    1.0
M5    1.0
M6    1.0
M7    1.0
M8    1.0
M9    1.0
Name: Importance, dtype: float64


Check if importance sums are approximately 1:


array([ True,  True,  True,  True,  True,  True,  True])

In [19]:
print("M9 feature importance from full audit:")
m9_from_audit = feature_importance_audit[
    feature_importance_audit["Model_ID"] == "M9"
].sort_values("Importance", ascending=False)

display(m9_from_audit)

print("\nM9 feature importance saved separately:")
display(final_m9_importance.sort_values("importance", ascending=False))

print("\nM9 importance sum from audit:")
print(m9_from_audit["Importance"].sum())

print("\nM9 importance sum from separate file:")
print(final_m9_importance["importance"].sum())

M9 feature importance from full audit:


,Model_ID,Model_name,Feature,Importance
21,M9,Absolute Healthcare & Environment Random Forest,gdp_per_capita,0.637233
22,M9,Absolute Healthcare & Environment Random Forest,health_expenditure_per_capita,0.195331
23,M9,Absolute Healthcare & Environment Random Forest,smoking,0.077840
24,M9,Absolute Healthcare & Environment Random Forest,air_pollution,0.052813
25,M9,Absolute Healthcare & Environment Random Forest,obesity,0.036783



M9 feature importance saved separately:


,feature,importance
0,gdp_per_capita,0.637233
1,health_expenditure_per_capita,0.195331
2,smoking,0.077840
3,air_pollution,0.052813
4,obesity,0.036783



M9 importance sum from audit:
0.9999999999999999

M9 importance sum from separate file:
0.9999999999999999


In [20]:
figures_path = "../reports/figures"

expected_figures = [
    "model_comparison_r2.png",
    "model_comparison_mae.png",
    "m9_feature_importance.png",
    "m8_vs_m9_feature_importance.png",
    "gdp_vs_life_expectancy.png",
    "log_gdp_vs_life_expectancy.png",
    "air_pollution_vs_life_expectancy.png"
]

print("Figures folder exists:", os.path.exists(figures_path))
print()

existing_figures = os.listdir(figures_path) if os.path.exists(figures_path) else []

for fig in expected_figures:
    print(fig, "->", fig in existing_figures)

print("\nAll files in reports/figures:")
print(existing_figures)

Figures folder exists: True

model_comparison_r2.png -> True
model_comparison_mae.png -> True
m9_feature_importance.png -> True
m8_vs_m9_feature_importance.png -> True
gdp_vs_life_expectancy.png -> True
log_gdp_vs_life_expectancy.png -> True
air_pollution_vs_life_expectancy.png -> True

All files in reports/figures:
['air_pollution_vs_life_expectancy.png', 'm8_vs_m9_feature_importance.png', 'model_comparison_mae.png', 'm9_feature_importance.png', 'model_comparison_r2.png', 'log_gdp_vs_life_expectancy.png', 'gdp_vs_life_expectancy.png']


In [21]:
audit_summary = {
    "base_dataset_rows": len(base_dataset),
    "final_model_dataset_rows": len(final_model_dataset),
    "base_potential_aggregates": len(base_aggregate_check),
    "final_potential_aggregates": len(final_aggregate_check),
    "final_missing_values_total": int(final_model_dataset.isna().sum().sum()),
    "model_count": len(model_audit),
    "rf_model_count": len(rf_performance_audit),
    "best_model_by_r2": model_audit.sort_values("R_squared", ascending=False).iloc[0]["Model_ID"],
    "best_r2": model_audit.sort_values("R_squared", ascending=False).iloc[0]["R_squared"],
    "best_model_by_mae": model_audit.sort_values("MAE_years", ascending=True).iloc[0]["Model_ID"],
    "best_mae": model_audit.sort_values("MAE_years", ascending=True).iloc[0]["MAE_years"],
    "m9_importance_sum": round(
        feature_importance_audit[
            feature_importance_audit["Model_ID"] == "M9"
        ]["Importance"].sum(),
        6
    )
}

audit_summary

{'base_dataset_rows': 212,
 'final_model_dataset_rows': 155,
 'base_potential_aggregates': 10,
 'final_potential_aggregates': 0,
 'final_missing_values_total': 0,
 'model_count': 9,
 'rf_model_count': 7,
 'best_model_by_r2': 'M9',
 'best_r2': np.float64(0.822),
 'best_model_by_mae': 'M4',
 'best_mae': np.float64(2.58),
 'm9_importance_sum': np.float64(1.0)}